# Lab 3: Word meaning & reasoning

The assignment covers two NLP tasks related to the word meaning and reasoning in a specialized domain.  
Part 1 deals with word meaning as it tackles the task of identifying the correct sense of a word in the context.  
Part 2 concerns reasoning with math question-answering problems and examines it from the generalization and explainability perspectives.

Shall we begin?

# Rules

You are greatly encouraged to add comments to your code describing what particular lines of code do (in general, a great habit to have in your coding life).
Additionally, please follow these rules when submitting the notebook:

1. Put all code in the cell with the `# YOUR CODE HERE` comment.
2. For theoretical questions, put your solution in the `YOUR ANSWER HERE` or `ANSWER UNDER THIS LINE` cells (and keep the header, if any).
3. Don't change or delete any initially provided cells, either text or code, unless explicitly instructed to do so.
4. Don't delete the comment lines `# TEST...` or edit their code cells. The test cells are for sanity checking. Passing them doesn't necessarily mean that your code is fine.
5. Don't change the names of the provided functions and variables, the arity of funcions, and types of function input/output.
7. Don't clear the output of your code cells.
8. Don't output unnecessary info (e.g., printing variables for debugging purposes). This clutters the notebook and slows down the grading. You can have print() in the code, but comment them out before submitting the notebook.
9. Delete those cells that you inserted for your own debugging/testing purposes.
10. When several lines of code is <font color="red">copied from GenAI</font>, mark them explicitly with commented tags `#<AI>\n code \n#</AI>`.
11. A full exercise solution that appears to be <font color="red">copied from GenAI</font> will receive 0 points.   
10. Don't forget to fill in the **contribution information**.
11. Don't forget to fill in the **work description section** per exercise.
12. Test your code and **make sure we can run your notebook** in the colab environment.
13. A single notebook file (without archiving) per group should be submitted via BrightSpace.

<font color="red">Following these rules helps us to grade the submissions relatively efficiently. If these rules are violated, a submission will be subject to penalty points. A serious breach of these rules in an exercise may result in 0 points.</font>  

# <font color="red">Contributions</font>

~~Delete this text and write instead of it yours:~~
* ~~a list of group members' names (NOT student IDs)~~
* ~~who contributed to which exercises and how. Note that it is important that each member has some contribution to each exercise, e.g., at least reviewing the solutions.~~

YOUR ANSWER HERE [30-50 words]

# General instructions

Before diving into the exercises, keep in mind that the variables defined previously can be reused in the subsequent cells. So there is no need to redefine the same variable in multiple sections, e.g., it is sufficient to read the file in a variable once and later reuse the value of the variable, instead of re-reading the file.   

Your code will often be evaluated based on its behaviour. So, during the grading some code cells are executed. If code runtime is too long than expected, this will hinder grading.

<font color="red">**Pay attention to test units**</font> that are either provided as assert cases or as comments. Test units help you by giving you a hint about the correct answer. Note that **passing test units doesn't guarantee the full points** for an exercise because test units are incomplete, and the code might fail on other test units.

# Part 1: Word sense disambiguation

Word Sense Disambiguation (WSD) is the task of identifying which meaning of a word is intended in a given context. For example, given an inventory of senses for "seal", one needs to identify a correct sense for each occurrence in the context:

* A <u>seal</u> was swimming near the icy shore.
* The document had an official <u>seal</u> at the bottom.
* Make sure the <u>seal</u> on the jar is not broken.

In this part we will use the BERT transformer model's contextualized word embeddings to tackle the WSD task. A sense inventory per word will be taken from WordNet. The approach consists of the following:

1. Get the contextualized BERT embeddings for all tokens in a sense-annotated corpus;
2. For each sense $s$, calculate a mean vector of all the vectors of the words that are tagged with the sense $s$ in the training part of the corpus;
3. For each token $t$ in the test corpus to which sense is applicable, assign $s$ sense to $t$ such that the vector of $s$ is the closest to the vector of $t$.
4. As a backup strategy for tokens in the test corpus for which no sense vector was obtained from the training part (i.e., tokens with unseen senses), use the 1st sense of the token by default.

## Setup

In [2]:
# Course-specific package
! rm -rf assigntools
! git clone https://github.com/kovvalsky/assigntools.git
from assigntools.NLP.deep_learning import transformer_word2convec
from assigntools.M4LP.A1 import read_pickle, write_pickle
! pip install svgling

Cloning into 'assigntools'...
remote: Enumerating objects: 271, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 271 (delta 30), reused 4 (delta 3), pack-reused 200 (from 1)
Receiving objects: 100% (271/271), 67.18 KiB | 1.27 MiB/s, done.
Resolving deltas: 100% (132/132), done.


In [3]:
import random, torch
torch.set_printoptions(precision=10)
# Tell PyTorch to use Tensor Cores for matrix multiplications and convolutions
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import torch.nn.functional as F
from collections import defaultdict, Counter
from tqdm import tqdm
from tabulate import tabulate
import nltk
from nltk.tree import Tree
from more_itertools import chunked
from nltk.corpus import wordnet as wn
from nltk.corpus import semcor
from nltk.corpus.reader.wordnet import Lemma
nltk.download('semcor')
nltk.download('wordnet')

# append any imports if needed

[nltk_data] Downloading package semcor to /home/david/nltk_data...
[nltk_data]   Package semcor is already up-to-date!
[nltk_data] Downloading package wordnet to /home/david/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## SemCor

As a sense annotated corpus, we will use SemCor, conveniently available within NLTK. <code>semcor.sents()</code> iterates over all sentences represented as lists of tokens, while <code>semcor.tagged_sents()</code> iterates over the same sentences with additional annotation including WordNet Lemma identifiers (Lemma in WordNet stands for a particular sense of a word as opposed to a synset that is a set of Lemmas).

Browsing WordNet with one of these online tools, [en-word.net](https://en-word.net/) and [visuwords](https://visuwords.com/), will help you to better/quicker understand the sense organization in WordNet.

In [4]:
# two sample sentence from the semcor corpus
# with their corresponding sense-annotated versions
for i in [0, 27]:
    print(semcor.sents()[i])
    print(semcor.tagged_sents(tag="sem")[i])

['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', 'Atlanta', "'s", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']
[['The'], Tree(Lemma('group.n.01.group'), [Tree('NE', ['Fulton', 'County', 'Grand', 'Jury'])]), Tree(Lemma('state.v.01.say'), ['said']), Tree(Lemma('friday.n.01.Friday'), ['Friday']), ['an'], Tree(Lemma('probe.n.01.investigation'), ['investigation']), ['of'], Tree(Lemma('atlanta.n.01.Atlanta'), ['Atlanta']), ["'s"], Tree(Lemma('late.s.02.recent'), ['recent']), Tree(Lemma('primary.n.01.primary_election'), ['primary', 'election']), Tree(Lemma('produce.v.04.produce'), ['produced']), ['``'], ['no'], Tree(Lemma('evidence.n.01.evidence'), ['evidence']), ["''"], ['that'], ['any'], Tree(Lemma('abnormality.n.04.irregularity'), ['irregularities']), Tree(Lemma('happen.v.01.take_place'), ['took', 'place']), ['.']]
['His', 'petition', 'charged', 'mental', 'cruelty

Let's prepare SemCor data for the disambiguation task. Since this is just an educational exercise and we don't aim at replicating the full results, we can use only a subset of SemCor. Let's take the first $N$ sentences of SemCor, pre-process the data, shuffle the sample in the data **randomly**, and finally split the data into the training and test sets.

In [5]:
# Extract a part of the data for experiments
N = 10_000
semcor_annotated = list(semcor.tagged_sents(tag='sem')[:N])
semcor_tokenized = list(semcor.sents()[:N])
random.Random(42).shuffle(semcor_annotated)
random.Random(42).shuffle(semcor_tokenized)

## Ex1.1 [7pt] Preprocessing data

Create a function that takes as input a collection of sense-annotated sentences from SemCor and extracts the sense annotation. For each token of the sentence get either the corresponding WordNet sense or <code>None</code>.
The latter is for the cases when:
1. A word token is not annotated with a Lemma object sense (e.g. articles and prepositions are such tokens);
2. There is no *straightforward* correspondance between a token and a sense annotation (see Q1.1 related to this).

More info about NLTK's Lemma and Tree objects can be found here: [Lemma](https://www.nltk.org/api/nltk.corpus.reader.wordnet.html) and [Tree](https://www.nltk.org/api/nltk.tree.tree.html).

In [34]:
def get_sns_annotations(data):
    """ data - sense tagged data from semcor
        return
            the sense annotations as a list of lists.
            The structure follows to semcor sentences and tokenization
            The elements of the list are None or a tuple of strings
            representing a synset and a lemma.
            None annotation means that a word token has no sense annotation
    """
    ### YOUR CODE HERE ###
    result = []

    for sentence in data:
        sentence_annotation = []
        # For each token $t$ in the test corpus to which sense is applicable, assign $s$ sense to $t$ such that the vector of $s$ is the closest to the vector of $t$.
        for token in sentence:
            if hasattr(token, 'label'):
                label = token.label()
                annotation = None
                # <AI>
                leaves = token.leaves()
                # <AI>
                print("LABEL: ", label)
                print("LEAVES: ", leaves)
                if len(leaves) == 1:
                    for subtree in token.subtrees():
                        label = subtree.label()

                        if hasattr(label, 'synset') and hasattr(label, 'name'):
                            try:

                                synset_name = label.synset().name()
                                lemma_name = label.name()
                                #print(synset_name, lemma_name)
                                annotation = (synset_name, lemma_name)
                                break
                            except(ValueError, AttributeError):
                                pass
                if annotation is not None:
                    sentence_annotation.append(annotation)
                else:
                    for _ in leaves:
                        sentence_annotation.append(None)
            else:
                for _ in token:
                    sentence_annotation.append(None)

        result.append(sentence_annotation)

    return result

In [35]:
# TEST Ex1.1
semcor_senses = get_sns_annotations(semcor_annotated)

print("sample sentence:", semcor_tokenized[0])
assert semcor_tokenized[0] == [
    'The',
    'bronchial',
    'artery',
    ',',
    'except',
    'for',
    'a',
    'small',
    'number',
    'of',
    'short',
    'branches',
    'in',
    'the',
    'hilum',
    ',',
    'contributes',
    'none',
    'of',
    'the',
    'pleural',
    'blood',
    'supply',
    '.'
]
print("sample annotation:", semcor_senses[0])
assert semcor_senses[0] == [
    None,
    None,
    None,
    None,
    None,
    None,
    None,
    ('small.a.01', 'small'),
    ('number.n.02', 'number'),
    None,
    ('short.a.02', 'short'),
    ('branch.n.03', 'branch'),
    None,
    None,
    ('hilus.n.01', 'hilum'),
    None,
    ('contribute.v.02', 'contribute'),
    None,
    None,
    None,
    ('pleural.a.01', 'pleural'),
    ('blood.n.01', 'blood'),
    ('supply.n.01', 'supply'),
    None
]

#print("sample sentence:", semcor_tokenized[13])
#print("sample annotation:", semcor_senses[13])

total_senses_id_data = len([ t for s in semcor_senses for t in s if t ])
print("Total number of senses in the data =", total_senses_id_data)
print("Ratio of senses in the data =", total_senses_id_data / sum([ len(s) for s in semcor_senses ]))

LABEL:  Lemma('bronchial_artery.n.01.bronchial_artery')
LEAVES:  ['bronchial', 'artery']
LABEL:  Lemma('small.a.01.small')
LEAVES:  ['small']
LABEL:  Lemma('number.n.02.number')
LEAVES:  ['number']
LABEL:  Lemma('short.a.02.short')
LEAVES:  ['short']
LABEL:  Lemma('branch.n.03.branch')
LEAVES:  ['branches']
LABEL:  Lemma('hilus.n.01.hilum')
LEAVES:  ['hilum']
LABEL:  Lemma('contribute.v.02.contribute')
LEAVES:  ['contributes']
LABEL:  Lemma('pleural.a.01.pleural')
LEAVES:  ['pleural']
LABEL:  Lemma('blood.n.01.blood')
LEAVES:  ['blood']
LABEL:  Lemma('supply.n.01.supply')
LEAVES:  ['supply']
LABEL:  Lemma('toe_the_line.v.01.toe_the_line')
LEAVES:  ['toe', 'the', 'line']
LABEL:  Lemma('be.v.01.be')
LEAVES:  ['was']
LABEL:  Lemma('child.n.01.child')
LEAVES:  ['child']
LABEL:  Lemma('excessively.r.01.too')
LEAVES:  ['too']
LABEL:  Lemma('much.r.01.much')
LEAVES:  ['much']
LABEL:  Lemma('part.n.01.part')
LEAVES:  ['part']
LABEL:  Lemma('environment.n.01.environment')
LEAVES:  ['environment

Reference output:
```
sample sentence: ['The', 'bronchial', 'artery', ',', 'except', 'for', 'a', 'small', 'number', 'of', 'short', 'branches', 'in', 'the', 'hilum', ',', 'contributes', 'none', 'of', 'the', 'pleural', 'blood', 'supply', '.']
sample annotation: [None, None, None, None, None, None, None, ('small.a.01', 'small'), ('number.n.02', 'number'), None, ('short.a.02', 'short'), ('branch.n.03', 'branch'), None, None, ('hilus.n.01', 'hilum'), None, ('contribute.v.02', 'contribute'), None, None, None, ('pleural.a.01', 'pleural'), ('blood.n.01', 'blood'), ('supply.n.01', 'supply'), None]
sample sentence: ['It', 'just', 'did', "n't", 'occur', 'to', 'Trig', 'that', 'anything', 'serious', 'would', 'happen', 'to', 'him', '.']
sample annotation: [None, ('merely.r.01', 'just'), None, None, ('occur.v.02', 'occur'), None, None, None, None, ('dangerous.s.02', 'serious'), None, ('happen.v.02', 'happen'), None, None, None]
```

## Q1.1 [3pt] Non-straightforward annotation

After exploring the data suffciently, tell all kinds of sense annotations that fall in the non-straightforward annotation category. For clarity, provide an example when describing each kind.
#TODO Finish the answer
- Multi-word expressions. By default, we had bronchial_artery but we want each word ot have a separate meaning if we
meet it alone.

<font color="red">█████ ANSWER UNDER THIS LINE [20-50 words] █████</font>

~~Delete this line and type your answer here, don't add a new text cell.~~

In [8]:
# create training and test sets
train_N = 9_000
semcor_X = {'train':semcor_tokenized[:train_N], 'test':semcor_tokenized[train_N:]}
semcor_Y = {'train':semcor_senses[:train_N], 'test':semcor_senses[train_N:]}

## BERT's contextualized vectors

After we have the training and test sets prepared with their gold sense annotations, it is time to get sense vectors for those senses that are occurring in the training set. Note that **contextualized vectors are crucial for the task** as a word (e.g., "book", "plant", "figure") can have different senses in different contexts.

We will use BERT transformer model to get contextualized word vectors for the words in the training and test sets. We will use the implementation of BERT in pytorch from the [transformers library](https://huggingface.co/docs/transformers/index).

Getting word vectors from BERT is not trivial as it uses a different type of tokenization than the traditional one. For example, the base-uncased version of BERT expects `Jupyter` tokenized as `ju`, `##py`, `##ter` while `Notebook` as `notebook` (note the lower casing of tokens due to the uncased version of BERT). To distinguish these two versions of tokenization and tokens, we will use `tokens` for BERT tokens and words for traditional tokenization. For example, the SemCor sentences use traditional tokenization.

If you want to learn more about BERT, [this](http://mccormickml.com/2019/05/14/BERT-word-embeddings-tutorial/) represents a gentle intro to BERT's wordpiece-based tokenizatio and contextualized word vectors.

In [9]:
# if this cell errors with "A UTF-8 locale is required. Got ANSI_X3.4-1968"
# uncomment and run the next two lines
import locale
locale.getpreferredencoding = lambda: "UTF-8"
# install transformer library
# !pip install transformers

In [10]:
import transformers
from transformers import BertModel, AutoTokenizer
print(transformers.__version__) # 5.0.0

/home/david/miniconda3/envs/NLP/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5.3.0


In [11]:
# Load tokenizer (vocabulary)
MODEL_NAME = 'bert-base-uncased'
bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [12]:
# As usual, tokens are mapped to indices
print("The size of the token vocabulary", len(bert_tokenizer.vocab))
for tok in ("dog", "##tion"):
    print(f"'{tok}' has index {bert_tokenizer.vocab[tok]}")

for i in (3899, 3508):
    print(f"Reverse mapping: {i} --> {bert_tokenizer.convert_ids_to_tokens(i)}")

The size of the token vocabulary 30522
'dog' has index 3899
'##tion' has index 3508
Reverse mapping: 3899 --> dog
Reverse mapping: 3508 --> ##tion


In [13]:
example_input = "Transformers in Jupyter Notebook"
tok_result = bert_tokenizer(example_input)
print("Output of a tokenizer: ", tok_result)
print("Tokens as word pieces: ", bert_tokenizer.convert_ids_to_tokens(tok_result.input_ids))

Output of a tokenizer:  {'input_ids': [101, 19081, 1999, 18414, 7685, 3334, 14960, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
Tokens as word pieces:  ['[CLS]', 'transformers', 'in', 'ju', '##py', '##ter', 'notebook', '[SEP]']


`[CLS]` and `[SEP]` are special tokens use by BERT. `[CLS]` gets a vector that models the meaning of the entire input text sequence while `[SEP]` indicates sequence delimiters. Note that one of the tasks BERT was pre-trained on was guessing the next sentence, hence it was trained on sequence modeling, where elements of the sequence are sentences. Note that the output of `tokenizer([S1, S2])` and `tokenizer(S1, S2)` differ as in the first case the input is interpreted as a batch of two independent texts while in the second it is a sequence of texts.

The output of the tokenizer provides a sufficient input for BERT to process the input and assign contextualized embeddings.

In [14]:
# Load pre-trained model (weights)
bert = BertModel.from_pretrained(MODEL_NAME)

#print parameters
total_params = sum(p.numel() for p in bert.parameters())
trainable_params = sum(p.numel() for p in bert.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2671.32it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters:     109,482,240
Trainable parameters: 109,482,240


In [15]:
# let bert output hidden states
bert.config.output_hidden_states = True
# bert expects tensors as an input
tok_result = bert_tokenizer(example_input, return_tensors='pt')
bert_output = bert(**tok_result)
print("Dimension of the last (12th) hidden states (batch size X token number X vector dim): ", bert_output.hidden_states[-1].shape)

Dimension of the last (12th) hidden states (batch size X token number X vector dim):  torch.Size([1, 8, 768])


Due to non-trivial correspondence between BERT tokens and words, we provide you with a ready function `transformer_word2convec` that takes a batch/list of word-tokenized sentences and returns BERT's contextualized embeddings for each word token. The embedding vectors of the words that consist of several tokens is obtained by collating token vectors (e.g., taking the mean by default). The function allows to indicate from which layer the vectors should be extracted. For more details you can read the function definition here: [transformer_word2convec](https://github.com/kovvalsky/assigntools/blob/main/NLP/deep_learning.py).

In [16]:
# Batches with GPU accelerates the process ~30 times when used T4 GPU of colab compared to CPU
# Obviously for this toy example, efficiency doesn't matter
# this identifies whether GPU is present
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("we are using", device)

print("Tokenizing and assigning embeddings:")
sample_batch = [ "Transformers in Jupyter Notebook".split(), "Transformers visited the earth".split() ]
sample_output = transformer_word2convec(bert, bert_tokenizer, sample_batch, device=device, collate_tok_vec=torch.mean, layer=-1)

# illustrating the output
for sent in sample_output:
    for w in sent:
        print(f"{w['word']:>20} ---> {str(w['tokens']):<20}: {w['pt'][:3]}...")
    print()

# Comparing vectors of two occurrences of "Transformers"
tvec1, tvec2 = sample_output[0][0]['pt'], sample_output[1][0]['pt']
print(f"{tvec1[:5]}... != {tvec2[:5]}...")
print(f"vectors cosine similarity = {F.cosine_similarity(tvec1, tvec2, dim=0)}")

we are using cuda
Tokenizing and assigning embeddings:
        Transformers ---> ['transformers']    : tensor([-0.2649184763, -0.0887792110, -0.1845795810])...
                  in ---> ['in']              : tensor([ 0.8251876831,  0.1082994714, -0.4692614079])...
             Jupyter ---> ['ju', '##py', '##ter']: tensor([-0.1717762649, -0.5189650059,  0.3763335347])...
            Notebook ---> ['notebook']        : tensor([-0.2266356945, -0.4329984188,  0.7514708042])...

        Transformers ---> ['transformers']    : tensor([-0.4978575110,  0.0919557288, -0.0247807465])...
             visited ---> ['visited']         : tensor([ 1.2025231123, -0.1833701879, -0.5396957994])...
                 the ---> ['the']             : tensor([-0.8388849497,  0.1043408588,  0.2743160129])...
               earth ---> ['earth']           : tensor([-0.8353101015, -0.2884499133, -0.1948720813])...

tensor([-0.2649184763, -0.0887792110, -0.1845795810,  0.4481189847,
         0.1534749269])... != te

## Ex1.2 [7pt] Sense vectors

Process the training set with BERT using `transformer_word2convec` with <font color="red">**default parameters**</font>. After getting word vectors, iterate over all train sentences, and for each sense, collect the word vectors. Note that words without sense annotations will be ignored in this process.

Since senses belonging to the same WordNet synset are by definition equivalent, we will be using synsets as sense labels. Prepare a dictionary with a synset string as a key and a single tensor as a value. The tensor should be a mean of all the word vectors collected for the synset (Note that the mean is a default collating method of `transformer_word2convec`).

This process is a time-consuming part of this assignment. It is recommended to use Colab's GPU: max 5min of T4 GPU will suffice to process all sentences with BERT. While processing the sentences, use batches of size 64 for consistency. Note that batches make a big difference with GPU. For the purposes of developing and debugging your solution, you may start by using a sample of 100 sentences, but then switch to the full training set.

<font color="red">Reuse global vars `bert` and `bert_tokenizer` for this and the next exercises.</font>

In [40]:
def get_sense2vec(data_X, data_Y, batch_size=64, device=device, collate=torch.mean):
    """ data_X and data_Y are a list of tokenized semcor sentences with their corresponding sense annotations.
        The last two arguments are the same as in transformer_word2convec
        Returns 2 dictionaries:
            Sense2VecList - { synset_str -> list of tensors  }
            Sense2AvgVec - { synset_str -> a mean tensor  }
    """
    Sense2VecList = defaultdict(list)
    Sense2AvgVec = {}

    word_vectors = []

    for idx in range(0, len(data_X), batch_size):
        batch_X = data_X[idx:idx + batch_size]

        batch_results = transformer_word2convec(bert, bert_tokenizer, batch_X, device=device)
        word_vectors.extend(batch_results)

    for sentences, annotations in zip(word_vectors, data_Y):
        for vector, annotation in zip(sentences, annotations):

            if annotation is not None:
                synset = annotation[0]

                Sense2VecList[synset].append(vector['pt'])

    for synset, vector_list in Sense2VecList.items():

        tensors = [v if isinstance(v, torch.Tensor) else torch.tensor(v, device=device) for v in vector_list]
        tensors = torch.stack(tensors)

        Sense2AvgVec[synset] = collate(tensors, dim=0)

    return Sense2AvgVec, Sense2VecList

In [41]:
%%time
# ~50min with CPU, less than 2min with T4 GPU
BATCH_SIZE = 64

Sense2AvgVec, Sense2VecList = get_sense2vec(semcor_X['train'], semcor_Y['train'], batch_size=BATCH_SIZE)

CPU times: user 1min 44s, sys: 6.92 s, total: 1min 51s
Wall time: 1min 45s


In [ ]:

# If you want to use reference vectors for next exercises, run the following:
# !rm -f Sense2AvgVec.pkl
# !wget -nv https://naturallogic.pro/_files_/download/mNLP/Sense2AvgVec.pkl
# Sense2AvgVec = read_pickle("Sense2AvgVec.pkl")

In [42]:
# TEST Ex1.2
for sns in [ 'mature.v.01', 'promptly.r.01', 'state.v.01', 'be.v.01']:
    assert isinstance(Sense2VecList[sns], list)
    assert isinstance(Sense2VecList[sns][0], torch.Tensor)
    assert isinstance(Sense2AvgVec[sns], torch.Tensor)
    print(f"{sns} sense has {len(Sense2VecList[sns])} vectors with the mean vector = {Sense2AvgVec[sns][:5]} ...")

mature.v.01 sense has 3 vectors with the mean vector = tensor([ 0.0585897565,  0.0471116118, -0.4077419043, -0.1625559628,
         0.2845914364]) ...
promptly.r.01 sense has 6 vectors with the mean vector = tensor([-0.2619910240,  0.0483156331,  0.1116875187,  0.0634896159,
         0.5795910954]) ...
state.v.01 sense has 410 vectors with the mean vector = tensor([ 0.3127556145,  0.2445111275,  0.0131903552, -0.0262363721,
         0.2175028026]) ...
be.v.01 sense has 2465 vectors with the mean vector = tensor([ 0.0335037448,  0.0922100917,  0.0208613947, -0.0746906027,
         0.2276506573]) ...


Reference output
```
mature.v.01 sense has 3 vectors...
promptly.r.01 sense has 6 vectors...
state.v.01 sense has 410 vectors...
be.v.01 sense has 2465 vectors...
```

## Ex1.3 [10pt] WSD testing

Now we are going to evaluate the sense embeddings on the test set. Write a function that takes a list of tokenized sentences and a mask that indicates which words are supposed to get senses. The function should return the sense predictions aligned with the sentences. When predicting a sense for a word token, use the strategy outlined above, with 1st WordNet sense as a fallback. Here is the strategy in more detail:

- Use the sense vectors that were calculated based on the training set;
- For each sense-annotated word token $t$ (e.g. the verb `run`) in the test set, predict the synset $s$ (e.g., `'run.x.xx'`) such that the vector of $s$ is the closest to the contextualized vector of $t$ based on the cosine distance metric `F.cosine_similarity`.
- For efficiency use [`wn.synsets()`](https://www.nltk.org/howto/wordnet.html) to narrow down possibel set of senses per word token.
- There will be word tokens $t$ in the test set for which there won't be a sense vector collected from the training set, i.e., unseen senses. For such word tokens, us a backup strategy and predict the 1st sense of the word from WordNet, which is the most common sense of the word. This can be done using a built-in function from NLTK (e.g. <code>wn.lemmas('run')[0]</code>). For more info about NLTK's WordNet API, check [this](https://www.nltk.org/howto/wordnet.html).

Note that the info that a word token has a gold sense unseen in the training set (provided by the mask argument), is not realistic info as it presupposes knowledge about gold annotations, but we are adopting this to simplify the task.

Below you are provided with the sense masking that indicates whether a word token gets a sense and whether its sense was seen in the training set.

In [ ]:
# masking of annotations: None - has NO sense annotation; True - has sense annotation and the sense
# was seen in the training set; False - has sense annotation but the sense was NOT seen in the training set
test_sense_mask = [ [ i if i is None else ( True if i[0] in Sense2AvgVec else False ) for i in s ] \
                        for s in semcor_Y['test'] ]

def wsd_accuracy(predictions, reference, verbose=False):
    """ Calculates accuracy with respect to the word tokens that get sense annotations.
    """
    true_and_false = []
    for preds, refs in zip(predictions, reference, strict=True):
        for (p, r) in zip(preds, refs, strict=True):
            if r is not None:
                true_and_false.append(p == r[0])
                if verbose and p != r[0]:
                    print(f"wrong prediction ({p}) for ({r})")
    return sum(true_and_false)/len(true_and_false)

In [ ]:
def predict_senses(sense2vec, sentences, sense_mask, fallback=False,
                   batch_size=BATCH_SIZE, bert=bert, tokenizer=bert_tokenizer, device=device, verbose=False):
    """ sense2vec - a dictioanry from synset strings to torch tensor vectors
        sentences - a list of sentences each being a list of word tokens
        sense_mask - it is aligned with tokens of sentences and tells if a token gets sense
                    and what type sense, seen or unseen in the training set.
        fallback - if True it uses first sense as the option for tokens with unseen senses.
        batch_size - the number of sentences is a batch
        bert, tokenizer - bert model and a tokenizer compatible with it
        device - a cpu or a gpu/cuda device that will be used by bert and tensor computations
        verbose - Prints whatever you want when it is False
        return predictions
            a list of list of predictions (None or a synset as a string) where the structure
            is aligned with the sentences
    """
    ### YOUR CODE HERE ###
    return predictions

In [ ]:
%%time
# TEST Ex2.3
# DON'T DELETE THE OUTPUT
# expected runtime less than a minute

predictions = predict_senses(Sense2AvgVec, semcor_X['test'], test_sense_mask)
predictions_fb = predict_senses(Sense2AvgVec, semcor_X['test'], test_sense_mask, fallback=True)

print("\nAccuracy of BERT      =", wsd_accuracy(predictions, semcor_Y['test']))
print("Accuracy of BERT + WN =", wsd_accuracy(predictions_fb, semcor_Y['test']))

Reference output (where `X<Y`):

```
Accuracy of BERT      = 0.6X...
Accuracy of BERT + WN = 0.6Y...
```

## Q1.3 [3pt] Error analysis

After exploring the predicted and gold senses, tell what type of tokens systematically get wrong sense labels?  
<small><small>Hint: reason behind it is the way we shortlist possible synsets per word token.</small></small>

<font color="red">█████ ANSWER UNDER THIS LINE [20-50 words] █████</font>

~~Delete this line and type your answer here, don't add a new text cell.~~

## Further experiments

Congratulations! You have reached the end of lab 4.

If you want an additional challenge, you can carry out further experiments on WSD task. Note that in the experiments we used the vectors from the last layer of BERT, but several research papers have shown that the vectors from different layers also encode useful information.

1.   You can test whether the vectors from other layers perform better than the last layer vectors.
2.   Word (and sense vectors) can be defined in terms of the combinations of the vectors from several layers of BERT. You can verify whether concatenating vectors from different layers (e.g., a concatenation of vectors from the last two layers) performs better than the vectors from the single layers.


## Work description for Part 1

~~Delete this and describe your approach to solving the exercises, for example, what steps you took first, followed by subsequent actions, which parts you found most challenging or easy, any specific helpful assistance received from TAs, whether you used GenAI, to what extent, at what stage, which one, how helpful was it, etc.~~

YOUR ANSWER HERE [100-200 words]

# Part 2: Generalization & explainability

In [ ]:
# @title imports
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
print(f"transformers={transformers.__version__}")
import torch
print(f"torch={torch.__version__}")
import datasets
print(f"datasets={datasets.__version__}")
from datasets import load_dataset
import re

## Task description

In this part, we will test one of the LLMs (which can be run in colab's modest environment) on the generalization and explainability while using chain-of-thought prompting ([Wei et al. 2022](https://arxiv.org/abs/2201.11903)).  
We will use an instruct model, a model that needs to be instructed with prompts to make predictions and doesn't need to be trained. So, we will use the model for inference and not for training or fine-tuning. Nonetheless, even Colab's least powerful GPU, such as T4, will be of practical use.  
Let's load an instruct model, in particular, `Phi-3-mini-4k-instruct` (see its [model card](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) for details), and make a processing pipeline out of it. We use this model as it is optimal for the Colab environment and has very good results.

In [ ]:
# @title model & tokenizer

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="auto", # Automatically place on GPU if available
    torch_dtype=torch.float16, # for good/efficient performance
    # trust_remote_code=True,
)
model.generation_config.max_length = 512

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# Create a pipeline
instruct_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,
    return_full_text=False, # don't return the input prompt
    do_sample = False # for determinism, generates next token greedily
)

## Ex2.1 [8pt] Data generation

We will use the GSM8K dataset ([Cobbe et al. 2021](https://arxiv.org/pdf/2110.14168)) from OpenAI to access simple arithmetic math problems.

In [ ]:
# @title GSM8K
GSM8K = load_dataset("openai/gsm8k", 'main')

In [ ]:
# we do some preprocessing, to get the short answer easily accessible
def preprocess_gsm8k(example):
    clean = re.sub(r'<<.*?>>', '', example['answer'])
    example['clean_answer'], example['short_answer'] = re.split(r'\s*\n####\s*', clean, maxsplit=1)
    return example

GSM8K = GSM8K.map(preprocess_gsm8k)

In [ ]:
# let's print a sample from the data
for k, v in GSM8K['train'][2025].items():
    print(f"{k}:\n{v}\n")

We need a prompt to give more elaborate instructions to the model. As the experiments show that the chain-of-thought (CoT) prompting boosts the model predictions, we are going to use it.  

Feel free to modify or add your messages below.

In [ ]:
# @title My prompt templates
# Chain-of-Thought prompt template

# we provide you with initial tempaltes but you should adapt them
# to get a better contrast between the two prompting approaches
MY_MESSAGES = {
    "no_cot":
        [{"role": "user",
          "content": "Answer the following problem by only providing the final answer:\nProblem: {problem}"}],
    "cot":
        [{"role": "user",
          "content": "Solve the following problem step by step and at the end give the final answer:\nProblem: {problem}"}],
}

In [ ]:
# Use template function from the native tokenizer instead of manually creating templates.
def message_to_template(message):
    # uses global var tokenizer
    return tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

print(message_to_template(MY_MESSAGES["cot"]))

Here we are defining the substitution rules. The idea behind the experiment is to check whether concept and number replacements irrelevant from the reasoning perspective will affect the model's predictions.  
If the model correctly answers a math question `q`:

> <font color="green"><small>At a bus station, a bus leaves every half-hour for 12 hours a day. How many buses leave the station for 5 days?</small></font>

then it should also correctly answer the math questions that are obtained from `q` with irrelevant concept/number substitutions.

> <font color="pink"><small>At a bus station, a bus leaves every <u>half-day</u> for <u>3</u> <u>days</u> a <u>month</u>. How many buses leave the station for <u>12</u> <u>months</u>?</small></font>

Otherwise it will indicate the poor generalization capacity of the model.

Below you have an illustration of how to define the concept and number replacement rules for particular QA problems.

In [ ]:
# @title demo substitutions
problem_subs = [
    (2025, {'concept': [
                {'question': ['bus -> train', 'buses -> trains'],                           'short_answer': '120'},
                {'question': ['leaves -> arrives', 'leaves the -> arrives in the'],         'short_answer': '120'},
                {'question': ['day -> month', 'days -> months', 'half-hour -> half-day', 'hours -> days'],    'short_answer': '120'}
                ],
            'number': [
                {'question': ['12 -> 5'],               'short_answer': '50'},
                {'question': ['12 -> 23'],              'short_answer': '230'},
                {'question': ['12 -> 3', '5 -> 12'],    'short_answer': '72'}
                ]
    }),
]

<font color="red">Pick **any two** QA problems from the test part of GSM8K.  
If your selected problems overlap with other groups' selected problems, this may be considered as plagiarism and will be subject to penalty.</font>

For example, the chances of two groups selecting the same two problems are extremely low, taking into account that the test split contains >1300 problems.  

In [ ]:
# @title My substitutions
# don't change the variable name!!!
MY_SUBS = [
    ### YOUR CODE HERE ###
    # here should be TWO(!) problem IDs from the GSM8K['test']
    # each with corresponding three(!) concept and three(!) number replacement rules
    # So in total, with the mix=True, this should define 2x(3+3+3x3)=30 new QA problems
    # make sure that the generated problems have the correct gold standard answers
]

In [ ]:
# @title generation function
def generate_prompted_samples(data, prob_subs, message=None, mix=True, v=False):
    """ The function takes data and a list of problem id and substitution rules.
        It returns a new list of QA problems that are obtained from the original ones
        by applying the substitutions.
        message is an optional flag telling the function to wrap new problems
        in a pre-defined template built from the message.
        mix tells the function to generate problems with a mixture of concept and number
        replacement rules. This option considers all possible combinations.
    """
    # a lits of (id, mode=concept|number|mix, new_question, new_short_answer, applied_subs)
    prompted_samples = []
    ### YOUR CODE HERE ###
    return prompted_samples

In [ ]:
# TEST generate_prompted_samples
# Let's see how the function should work
prompted_samples = generate_prompted_samples(
    GSM8K['train'], problem_subs, message=MY_MESSAGES["cot"])
print(f"total samples generated = {len(prompted_samples)}")

# print a sample generated problem
print(prompted_samples[-1][1])
print(prompted_samples[-1][-1])
print(prompted_samples[-1][2])
print(f"Answer = {prompted_samples[-1][-2]}")

The reference output of the above cell
```
total samples generated = 15
mix
['day -> month', 'days -> months', 'half-hour -> half-day', 'hours -> days', '12 -> 3', '5 -> 12']
<|user|>
Solve the following problem step by step and at the end give the final answer:
Problem: At a bus station, a bus leaves every half-day for 3 days a month. How many buses leave the station for 12 months?<|end|>
<|assistant|>

Answer = 72
```

## Ex2.2 [12pt] Evaluation

Now let's see how the model can be fed with an input and get predictions out of it.  
We will illustrate the both options, with the CoT template and without it.

In the end, you will have to run the instruct model on the generated samples with and without CoT prompts.

In [ ]:
# @title demo inference

# note that the correct answer is 72
only_problem = '''At a bus station, a bus leaves every half-day for 3 days a month. How many buses leave the station for 12 months?'''
# only answer prompt
OA_prompt = message_to_template(MY_MESSAGES["no_cot"]).format(problem=only_problem)
# CoT prompt
CoT_prompt = message_to_template(MY_MESSAGES["cot"]).format(problem=only_problem)


print(f"{'OA prompt':-^80}\n{OA_prompt}\n{'':-^80}")
output = instruct_pipe(OA_prompt)
print(output[0]['generated_text'])
print(f'{"":=^80}')

print(f"{'CoT prompt':-^80}\n{CoT_prompt}\n{'':-^80}")
output = instruct_pipe(CoT_prompt)
print(output[0]['generated_text'])

Use the below code cell to run your experiment with CoR and without CoT:
* On the selected probelms
* With the designed concept/number/mix substitutions
* With your own prompt templates, with and w/o CoT.

<font color="red">If the model continues to generate explanations under the non-CoT setting, your task is to develop a prompt template that reduces these unprompted explanations as much as possible. This effort will be count against the points.</font>


In [ ]:
# @title My inference
# Run the model on the generated problems with and without CoT prompting
# Feel free to include the problem generation script here too for no CoT prompting
# it takes 15-20min on gpu T4

def cot_vs_nocot(my_subs, cot_message, no_cot_message, verbose=False):
    """ Run the cot and no_cot experiments on the selected
        problems from GSM8K test and on their variants based
        on the manually defined substitutions from my_subs.
        Use corresponding prompt templates for cot and no_cot.
        Return a dictionary with the following structure:
        {'cot': [run1,...], 'no_cot': [run1,...]}
        where each run is a tuple (prob_id, gold_answer, mode, model_response)
        with mode being 'concept', 'number', 'mix'.
        Global fixed variables, such as data and models, and
        the defined functions are supposed to be reused here.
    """
### YOUR CODE HERE ###

    return answers

In [ ]:
%%time
answers = cot_vs_nocot(MY_SUBS, MY_MESSAGES["cot"], MY_MESSAGES["no_cot"])

In [ ]:
# TEST
for c in ["no_cot", "cot"]:
    print(f"{c.upper():#^80}")
    for pid, ans, mode, resp in answers[c]:
        title = f"  {cot}  {pid}   {cot}   {mode}   ans[{ans}]  "
        print(f"{'':-^70}")
        print(f"{title:-^70}\n\n{resp}")

### Evaluation results

Manually verify the predictions and fill in the table below based on the results.  
The 0s should be replaced with the number of correct predictions. For Concept and Number types, this is a maximum of 3, as the total number of generated samples is 3 per type. For the Mix type, it will be between 0 and 9.
You are also required to provide an evaluation on the original problems too.

You are expected to manually fill in the first table with correct counts because you will have to **manually assess the correctness of CoT reasoning/explanation**. Expl.+Ans means that **both** explanation text and the answer should be correct.
It is recommended that group members divide the assessment task among each other.

Note that counting correct answers is as simple as one needs to manually spot the final numerical answer and comparing it to the correct number. It is important to print the results in a way that will make it easy for you to assess explanations and the numerical answers.

<font color="red">█████ FILL IN THE TABLE █████</font>

<font color="red">Replace the dummy numbers while preserving the formatting.</font>  

---
### Table for the model with CoT

<table style="border: 1px solid black; border-collapse: separate; width: 100%;">
  <thead>
    <tr style="text-align: center; border: 1px solid black;">
      <th style="border: 1px solid black;">pro id</th>
      <th style="border: 1px solid black; width: 200px;">question</th>
      <th style="border: 1px solid black;">Ans</th>
      <th style="border: 1px solid black;" colspan="2">Original (1)</th>
      <th style="border: 1px solid black;" colspan="2">Concept (3)</th>
      <th style="border: 1px solid black;" colspan="2">Number (3)</th>
      <th style="border: 1px solid black;" colspan="2">Mix (9)</th>
    </tr>
    <tr style="text-align: center; border: 1px solid black;">
      <th style="border: 1px solid black;"></th>
      <th style="border: 1px solid black;"></th>
      <th style="border: 1px solid black;"></th>
      <th style="border: 1px solid black;">Ans</th>
      <th style="border: 1px solid black;">Expl+Ans</th>
      <th style="border: 1px solid black;">Ans</th>
      <th style="border: 1px solid black;">Expl+Ans</th>
      <th style="border: 1px solid black;">Ans</th>
      <th style="border: 1px solid black;">Expl+Ans</th>
      <th style="border: 1px solid black;">Ans</th>
      <th style="border: 1px solid black;">Expl+Ans</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>2025</td>
      <td style="width: 200px;">At a bus station, a bus leaves every half-hour for 12 hours a day.<br>How many buses leave the station for 5 days?</td>
      <td>120</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <td>2026</td>
      <td style="width: 200px;">Inez has $150. She spends one-half on hockey skates and a certain<br>amount on hockey pads.If Inez has $25 remaining, how much did<br>the hockey pads cost, together, in dollars?</td>
      <td>50</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
  </tbody>
</table>


---
### Table for the model with <font color="red">No</font> CoT

<table style="border: 1px solid black; border-collapse: separate; width: 100%;">
  <thead>
    <tr style="text-align: center; border: 1px solid black;">
      <th style="border: 1px solid black;">pro id</th>
      <th style="border: 1px solid black; width: 200px;">question</th>
      <th style="border: 1px solid black;">Ans</th>
      <th style="border: 1px solid black;">Original (1)</th>
      <th style="border: 1px solid black;">Concept (3)</th>
      <th style="border: 1px solid black;">Number (3)</th>
      <th style="border: 1px solid black;">Mix (9)</th>
    </tr>
    <tr style="text-align: center; border: 1px solid black;">
      <th style="border: 1px solid black;"></th>
      <th style="border: 1px solid black;"></th>
      <th style="border: 1px solid black;"></th>
      <th style="border: 1px solid black;">Ans</th>
      <th style="border: 1px solid black;">Ans</th>
      <th style="border: 1px solid black;">Ans</th>
      <th style="border: 1px solid black;">Ans</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>2025</td>
      <td style="width: 200px;">At a bus station, a bus leaves every half-hour for 12 hours a day.<br>How many buses leave the station for 5 days?</td>
      <td>120</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <td>2026</td>
      <td style="width: 200px;">Inez has $150. She spends one-half on hockey skates and a certain<br>amount on hockey pads.If Inez has $25 remaining, how much did<br>the hockey pads cost, together, in dollars?</td>
      <td>50</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
  </tbody>
</table>

## Discussion

Based on the results in the table, discuss:

1. The generalization capacity of both models,
2. To what extent the models followed the prompts (e.g., was any explanation generated in case of no_CoT?)
3. The contribution of the CoT prompting (i.e., contrasting only answers of CoT and noCoT models)
4. The alignment of the explanations with the predicted answers.

<font color="red">█████ ANSWER UNDER THIS LINE [100-200 words] █████</font>

~~Delete this line and type your answer here, don't add a new text cell.~~

## Work description for Part 2

~~Delete this and describe your approach to solving the exercises, for example, what steps you took first, followed by subsequent actions, which parts you found most challenging or easy, any specific helpful assistance received from TAs, whether you used GenAI, to what extent, at what stage, which one, how helpful was it, etc.~~

YOUR ANSWER HERE [100-200 words]

# Acknowledgments

The initial version of Part 1 by Denis Paperno was a replication of the WSD experiment for ELMo. The assignment was built around the allennlp library.
Since 2022-23 course, the assignment was substantially changed by Lasha Abzianidze. allennlp was replaced with pytorch and transformers library. ELMo was replaced with BERT.

Part 2 is a new addition to 2024-25 notebook, by Lasha Abzianidze.